# 01 — Bronze Ingestion

Reads raw Parquet files from disk and loads them into the Bronze layer in MongoDB (tlc_bronze). Metadata tags (_meta) are injected at this stage. The ControlManager records the full execution lifecycle.

## Configuration

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, write_mongo, read_parquet
from src.paths import list_raw_files, str_path
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
import pyspark.sql.functions as F
import uuid

VEHICLE_TYPES = ["yellow", "green", "fhv", "hvfhv"]
YEARS         = [2023, 2024, 2025]

print("Configuration loaded.")

Configuration loaded.


## Start Spark Session

In [2]:
spark = get_spark(app_name="TLC_Bronze_Ingestion")
print(f"Spark version: {spark.version}")

2026-07-18 21:42:08 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
Spark version: 3.4.1


## Run Bronze Ingestion Pipeline

In [ ]:
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

control = ControlManager("bronze_ingestion")
execution = control.start({"vehicle_types": VEHICLE_TYPES, "years": YEARS})

total_written = 0

for vehicle_type in VEHICLE_TYPES:
    files = list_raw_files(vehicle_type, years=YEARS)
    if not files:
        print(f"[WARN] No files found for vehicle_type='{vehicle_type}'. Skipping.")
        continue

    print(f"\n[{vehicle_type.upper()}] Processing {len(files)} file(s)...")

    for parquet_file in tqdm(files, desc=vehicle_type.upper(), unit="file"):
        run_id = execution.execution_id

        # Extraccion instantanea de metadata (O(1)) para evitar df.count() costosos
        try:
            num_rows = pq.ParquetFile(str_path(parquet_file)).metadata.num_rows
        except Exception:
            num_rows = 0

        df = read_parquet(spark, str_path(parquet_file))

        # Inject metadata
        df = df.withColumn("_meta", F.struct(
            F.lit(vehicle_type).alias("vehicle_type"),
            F.lit(run_id).alias("run_id"),
            F.current_timestamp().alias("ingestion_time"),
            F.lit(parquet_file.name).alias("source_file"),
        ))

        # Insertar. Le pasamos num_rows para auditar y que write_mongo no haga df.count()
        n = write_mongo(df, settings.MONGO_DB_BRONZE, f"{vehicle_type}_raw", mode="append", num_rows=num_rows)
        total_written += n

control.end(
    ExecutionStatus.SUCCESS,
    {"total_records_written": total_written, "vehicle_types": VEHICLE_TYPES},
)
print(f"\nBronze ingestion complete. Total records: {total_written:,}")

2026-07-18 21:42:09 | INFO     | tlc.audit.control_manager | [AUDIT] Execution started | pipeline=bronze_ingestion id=e0780a75

[YELLOW] Processing 36 file(s)...


YELLOW:   0%|          | 0/36 [00:00<?, ?file/s]

2026-07-18 21:42:10 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/yellow/yellow_tripdata_2023-01.parquet (Robust Mode)
2026-07-18 21:42:38 | INFO     | tlc.spark_utils | [SPARK] Wrote 3,066,766 rows → MongoDB tlc_bronze.yellow_raw (mode=append)
2026-07-18 21:42:38 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/yellow/yellow_tripdata_2023-02.parquet (Robust Mode)
2026-07-18 21:43:03 | INFO     | tlc.spark_utils | [SPARK] Wrote 2,913,955 rows → MongoDB tlc_bronze.yellow_raw (mode=append)
2026-07-18 21:43:03 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/yellow/yellow_tripdata_2023-03.parquet (Robust Mode)
2026-07-18 21:43:29 | INFO     | tlc.spark_utils | [SPARK] Wrote 3,403,766 rows → MongoDB tlc_bronze.yellow_raw (mode=append)
2026-07-18 21:43:29 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/yellow/yellow_tripdata_2023-04.parquet (Robust Mode)
2026-0

GREEN:   0%|          | 0/36 [00:00<?, ?file/s]

2026-07-18 21:52:42 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/green/green_tripdata_2023-01.parquet (Robust Mode)
2026-07-18 21:52:43 | INFO     | tlc.spark_utils | [SPARK] Wrote 68,211 rows → MongoDB tlc_bronze.green_raw (mode=append)
2026-07-18 21:52:43 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/green/green_tripdata_2023-02.parquet (Robust Mode)
2026-07-18 21:52:44 | INFO     | tlc.spark_utils | [SPARK] Wrote 64,809 rows → MongoDB tlc_bronze.green_raw (mode=append)
2026-07-18 21:52:44 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/green/green_tripdata_2023-03.parquet (Robust Mode)
2026-07-18 21:52:45 | INFO     | tlc.spark_utils | [SPARK] Wrote 72,044 rows → MongoDB tlc_bronze.green_raw (mode=append)
2026-07-18 21:52:45 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/green/green_tripdata_2023-04.parquet (Robust Mode)
2026-07-18 21:52:45 | INFO

FHV:   0%|          | 0/36 [00:00<?, ?file/s]

2026-07-18 21:53:10 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/fhv/fhv_tripdata_2023-01.parquet (Robust Mode)
2026-07-18 21:53:16 | INFO     | tlc.spark_utils | [SPARK] Wrote 1,114,320 rows → MongoDB tlc_bronze.fhv_raw (mode=append)
2026-07-18 21:53:16 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/fhv/fhv_tripdata_2023-02.parquet (Robust Mode)
2026-07-18 21:53:22 | INFO     | tlc.spark_utils | [SPARK] Wrote 1,110,797 rows → MongoDB tlc_bronze.fhv_raw (mode=append)
2026-07-18 21:53:22 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/fhv/fhv_tripdata_2023-03.parquet (Robust Mode)
2026-07-18 21:53:30 | INFO     | tlc.spark_utils | [SPARK] Wrote 1,328,242 rows → MongoDB tlc_bronze.fhv_raw (mode=append)
2026-07-18 21:53:30 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/fhv/fhv_tripdata_2023-04.parquet (Robust Mode)
2026-07-18 21:53:36 | INFO     | tlc.sp

HVFHV:   0%|          | 0/36 [00:00<?, ?file/s]

2026-07-18 21:57:23 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/hvfhv/fhvhv_tripdata_2023-01.parquet (Robust Mode)
2026-07-18 22:00:49 | INFO     | tlc.spark_utils | [SPARK] Wrote 18,479,031 rows → MongoDB tlc_bronze.hvfhv_raw (mode=append)
2026-07-18 22:00:49 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/hvfhv/fhvhv_tripdata_2023-02.parquet (Robust Mode)
2026-07-18 22:04:05 | INFO     | tlc.spark_utils | [SPARK] Wrote 17,960,971 rows → MongoDB tlc_bronze.hvfhv_raw (mode=append)


## Audit Report

In [ ]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))